# 🧲 chemsplit `similarity` splitters: fingerprint-clustering holdouts

Welcome! This notebook is a hands-on tour of chemsplit's **`similarity` splitter family** — 10 strategies that all reason about **pairwise fingerprint similarity** (Tanimoto/Dice over a molecular fingerprint, or any precomputed feature matrix) to cluster, threshold, or greedily select records so that train/test chemical space ends up deliberately close, deliberately far apart, or partitioned into similarity-based folds.

Reach for this family whenever a scaffold split feels too easy (or too coarse) and you want the holdout difficulty to track *how similar the fingerprints actually are*, not just whether two molecules share a Bemis-Murcko framework.

<a id="0"></a>
### ⚙️ Setup — the shared fixture

Every example below reuses the same tiny 20-molecule set, split into two chemically distinct families — aliphatic alkanes/alcohols vs. aromatic amines — so each splitter's clustering behaviour is easy to see at a glance. `summarize()` just prints the resulting partition sizes and a couple of interesting metadata fields.

In [1]:
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

import numpy as np

from chemsplit.splitters.similarity import (
    BalancedMultiTaskSplitter,
    ButinaSplitter,
    DensityClusterSplitter,
    KMeansClusterSplitter,
    LeaveOneClusterOutSplitter,
    MaxDissimilaritySplitter,
    MaxMinSplitter,
    PerimeterSplitter,
    SimilarityThresholdSplitter,
    SpectralSplitter,
)

_FAMILY_A = [
    "CCCCCC", "CCCCCCC", "CCCCCCCC", "CCCCCCCCC", "CCCCCCCCCC",
    "CCCCCCO", "CCCCCCCO", "CCCCCCCCO", "CCCCCCCCCO", "CCCCCCCCCCO",
]
_FAMILY_B = [
    "c1ccc(N)cc1", "c1ccc(N)nc1", "c1ccc(N)cc1C", "c1ccc(N)cc1CC", "Cc1ccc(N)cc1",
    "c1ccc2[nH]ccc2c1", "c1ccc2ncccc2c1", "c1ccc2[nH]ncc2c1", "Nc1ccc2ccccc2c1", "Nc1ccc2[nH]ccc2c1",
]
SMILES_20 = _FAMILY_A + _FAMILY_B


def summarize(result, label):
    print(f"[{label}] train={len(result.train)} valid={len(result.valid)} "
          f"test={len(result.test)} discard={len(result.discard)}")
    interesting = {k: v for k, v in result.metadata.items()
                   if not isinstance(v, (list, dict)) or len(str(v)) < 80}
    if interesting:
        print("  metadata:", interesting)

<a id="0.1"></a>
### ⚗️ `featurizer` and `metric` — the choice that defines "similar"

Every splitter below except `LeaveOneClusterOutSplitter` and `BalancedMultiTaskSplitter` (which delegate to a caller-supplied `clusterer` instead) takes two shared, keyword-only parameters that matter more than any other knob in this notebook:

- **`featurizer`** (default `"ecfp4"`) — how each molecule becomes a feature vector. String aliases: `ecfp2`/`ecfp4`/`ecfp6`/`ecfp8` (`morgan2`/`morgan3` are synonyms), `fcfp2`/`fcfp4`/`fcfp6`/`fcfp8`, `maccs`, `rdkitfp`, `avalon`, `atompair`, `toptorsion` (all binary fingerprints), the continuous-valued `physchem` and `mqn` descriptor sets, or `precomputed` to hand in your own feature matrix (`PrecomputedFeaturizer(X=...)`). An already-instantiated `Featurizer` object works too.
- **`metric`** (default `"tanimoto"`) — the pairwise distance over that feature vector: `tanimoto`, `dice`, `cosine`, `tanimoto_count` are bounded in `[0, 1]`; `euclidean`/`manhattan` are not. Splitters that treat `cutoff`/`threshold` as a *similarity* (`SimilarityThresholdSplitter`, `ButinaSplitter`) require a bounded metric and raise `ParameterError` otherwise.

Both are keyword-only on every constructor below, e.g. `ButinaSplitter(cutoff=0.3, featurizer="maccs", metric="dice")`. The default `ecfp4`/`tanimoto` pair is a reasonable starting point, but it is **not neutral** — swapping either one changes which molecules count as neighbours, and therefore changes the split.

In [2]:
from chemsplit.featurizers import get_featurizer

for alias in ["ecfp4", "maccs", "avalon", "atompair", "physchem", "mqn"]:
    feat = get_featurizer(alias)
    print(f"{alias:10s} -> {type(feat).__name__:20s} n_features={feat.n_features:5d}  binary={feat.is_binary}")

ecfp4      -> ECFPFeaturizer       n_features= 2048  binary=True
maccs      -> MACCSFeaturizer      n_features=  167  binary=True
avalon     -> AvalonFeaturizer     n_features= 1024  binary=True
atompair   -> AtomPairFeaturizer   n_features= 2048  binary=True
physchem   -> PhysChemFeaturizer   n_features=   12  binary=False
mqn        -> MQNFeaturizer        n_features=   42  binary=False


The same 20-molecule fixture, clustered by `ButinaSplitter` under three different `featurizer`/`metric` pairs — cluster membership is not remotely the same split, even though every other parameter is identical:

In [3]:
for featurizer, metric, cutoff in [
    ("ecfp4", "tanimoto", 0.4),
    ("maccs", "dice", 0.3),
    ("atompair", "cosine", 0.3),
]:
    splitter = ButinaSplitter(cutoff=cutoff, featurizer=featurizer, metric=metric, train_size=0.5, test_size=0.5, random_state=0)
    groups = splitter.compute_groups(SMILES_20)
    print(f"featurizer={featurizer:10s} metric={metric:8s} cutoff={cutoff}  "
          f"n_clusters={len(set(groups.tolist()))}  groups={groups.tolist()}")

featurizer=ecfp4      metric=tanimoto cutoff=0.4  n_clusters=11  groups=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
featurizer=maccs      metric=dice     cutoff=0.3  n_clusters=5  groups=[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 4, 3, 2, 3]
featurizer=atompair   metric=cosine   cutoff=0.3  n_clusters=6  groups=[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 3, 2, 4, 5, 3, 3, 3, 2, 3]


<a id="1"></a>
## 1. 🧲 Similarity splitters

<a id="1.1"></a>
### 1.1 🚧 `SimilarityThresholdSplitter` — a hard cross-similarity ceiling

Hard constraint: no test record may exceed `threshold` similarity to any train record — the constraint is explicit, checkable, and reported.

> 💡 **Advantages:**
> - The constraint is explicit, checkable, and reported — `metadata["max_cross_similarity"]` is either below the threshold or the split is wrong.
> - Directly parameterises what people actually mean by "novel chemistry": how dissimilar must test compounds be?
> - `graph_component` never discards data, so the full dataset gets used.

> ⚠️ **Pitfalls:**
> - **The threshold is the experiment.** The same cutoff means wildly different things across fingerprints/metrics — a result without fingerprint, radius, metric, and cutoff isn't reproducible.
> - Similarity isn't transitive, so connected components can be enormous and chemically incoherent; on dense datasets one component can swallow everything (raised as `ConstraintUnsatisfiableError`).
> - `greedy_prune`/`seeded_growth` discard records — exactly the ones in the interesting boundary region.

| Parameter | Meaning |
|---|---|
| `threshold` | similarity cutoff no test record may exceed vs. any train record |
| `strategy` | how the constraint is enforced: `graph_component` / `greedy_prune` / `seeded_growth` |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance/similarity, e.g. `tanimoto` (default), `dice` — must be bounded in `[0, 1]` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [4]:
splitter = SimilarityThresholdSplitter(
    threshold=0.3, strategy="graph_component", train_size=0.5, test_size=0.5, random_state=0
)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "SimilarityThresholdSplitter")

[SimilarityThresholdSplitter] train=10 valid=0 test=10 discard=0
  metadata: {'realised_sizes': {'train': 10, 'valid': 0, 'test': 10}, 'threshold': 0.3, 'strategy': 'graph_component', 'n_components': 2, 'max_cross_similarity': 0.0}


`graph_component` guarantees `max_cross_similarity` stays below 0.3 by construction — worth checking in the metadata above.

<a id="1.2"></a>
### 1.2 🔵 `ButinaSplitter` — Taylor-Butina sphere-exclusion clustering

Taylor-Butina sphere-exclusion (leader) clustering — a solid, chemically meaningful default that's markedly harder than a scaffold split.

> 💡 **Advantages:**
> - A solid default: harder than a scaffold split, chemically meaningful, and cheap enough for tens of thousands of molecules.
> - Deterministic, with no seed and only one interpretable parameter (`cutoff`) to choose.
> - Every cluster centroid is a real molecule, so clusters can be inspected and reported.

> ⚠️ **Pitfalls:**
> - Membership is defined only relative to the **centroid**, not pairwise — two cluster members can be up to `2 x cutoff` apart.
> - Produces many singletons on diverse libraries (often 30-60% of records); `singleton_policy` materially changes difficulty.
> - Highly sensitive to `cutoff` — small changes can halve or double the cluster count.

| Parameter | Meaning |
|---|---|
| `cutoff` | sphere-exclusion radius (distance by default, see `cutoff_is`) |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance/similarity, e.g. `tanimoto` (default), `dice` — must be bounded in `[0, 1]` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [5]:
splitter = ButinaSplitter(cutoff=0.4, train_size=0.5, test_size=0.5, random_state=0)
groups = splitter.compute_groups(SMILES_20)
result = splitter.split_result(SMILES_20)[0]
print("cluster labels:", groups.tolist())
summarize(result, "ButinaSplitter")

cluster labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[ButinaSplitter] train=10 valid=0 test=10 discard=0
  metadata: {'realised_sizes': {'train': 10, 'valid': 0, 'test': 10}, 'cutoff': 0.4, 'cutoff_is': 'distance', 'n_clusters': 11, 'cluster_sizes': [10, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'centroids': [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]}


The cluster labels above line up with the two families we built the fixture from — Butina recovers them without being told the family boundary.

<a id="1.3"></a>
### 1.3 🎯 `KMeansClusterSplitter` — partitional clustering that scales

K-means (or a related partitional clusterer) over fingerprint/feature space — scales far better than any `O(n²)` method.

> 💡 **Advantages:**
> - Scales far better than any `O(n²)` method — `minibatch_kmeans` handles millions of molecules.
> - The cluster count is an explicit, reportable knob, and `auto_rule` makes the default reproducible.
> - Works on any feature representation, including learned embeddings and physicochemical descriptors.

> ⚠️ **Pitfalls:**
> - **k is arbitrary** — nothing in the chemistry determines it, yet split difficulty depends on it strongly.
> - K-means assumes isotropic, roughly equal-variance clusters, which binary fingerprint space isn't.
> - Cluster sizes come out wildly uneven, so the achieved train/test ratio drifts from the request.

| Parameter | Meaning |
|---|---|
| `n_clusters` | number of k-means clusters to form |
| `featurizer` | feature representation — works with any of them, including continuous `physchem`/`mqn` descriptors; see [§0.1](#0.1) |
| `metric` | distance used for `reduce_dim`/clustering prep, e.g. `tanimoto` (default), `euclidean` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [6]:
splitter = KMeansClusterSplitter(n_clusters=2, train_size=0.5, test_size=0.5, random_state=0)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "KMeansClusterSplitter (ecfp4/tanimoto)")

# Same clusterer, continuous physchem descriptors instead of a binary fingerprint —
# demonstrates the "works on any feature representation" claim above.
splitter_physchem = KMeansClusterSplitter(
    n_clusters=2, featurizer="physchem", metric="euclidean", train_size=0.5, test_size=0.5, random_state=0
)
result_physchem = splitter_physchem.split_result(SMILES_20)[0]
summarize(result_physchem, "KMeansClusterSplitter (physchem/euclidean)")

[KMeansClusterSplitter (ecfp4/tanimoto)] train=10 valid=0 test=10 discard=0
  metadata: {'realised_sizes': {'train': 10, 'valid': 0, 'test': 10}, 'n_clusters': 2, 'algorithm': 'kmeans', 'nondeterministic_method': True}
[KMeansClusterSplitter (physchem/euclidean)] train=14 valid=0 test=6 discard=0
  metadata: {'realised_sizes': {'train': 14, 'valid': 0, 'test': 6}, 'n_clusters': 2, 'algorithm': 'kmeans', 'nondeterministic_method': True}


⚠️ Note the `SizeToleranceWarning` on the `physchem` run: raw physchem descriptors (molecular weight, TPSA, ...) live on wildly different numeric scales, so an unscaled `euclidean` distance is dominated by whichever descriptor happens to have the largest range — always standardise continuous features (or use `metric="cosine"`) before clustering on them.

<a id="1.4"></a>
### 1.4 🌫️ `DensityClusterSplitter` — DBSCAN/HDBSCAN with an explicit noise bucket

DBSCAN or HDBSCAN density clustering on a precomputed distance matrix — no `k` to choose, and clusters can take any shape.

> 💡 **Advantages:**
> - No `k` to choose, and clusters can take any shape — a better match for chemical space than k-means's spherical assumption.
> - Explicitly models "this molecule belongs to no family", which every other clusterer here forces into some cluster.
> - `noise_policy="test"` produces a clean, defensible "singletons and oddities" test set for applicability-domain work.

> ⚠️ **Pitfalls:**
> - The noise bucket can swallow a large fraction of a diverse library (40%+ at sensible `eps`), and `noise_policy` then decides most of the split.
> - `eps` interacts with fingerprint density in a way with no cross-dataset meaning — a value tuned on one dataset doesn't transfer.
> - Needs `O(n²)` memory on a precomputed matrix, capping `n` around 20,000 at the default guard.

| Parameter | Meaning |
|---|---|
| `algorithm` | clustering backend: `dbscan` or `hdbscan` |
| `min_cluster_size` | minimum points to form a (non-noise) cluster |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance, e.g. `tanimoto` (default), `euclidean` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [7]:
splitter = DensityClusterSplitter(
    algorithm="hdbscan", min_cluster_size=2, train_size=0.5, test_size=0.5, random_state=0
)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "DensityClusterSplitter (hdbscan)")

[DensityClusterSplitter (hdbscan)] train=10 valid=0 test=10 discard=0
  metadata: {'realised_sizes': {'train': 10, 'valid': 0, 'test': 10}, 'algorithm': 'hdbscan', 'n_clusters': 3, 'n_noise': 0, 'noise_frac': 0.0, 'noise_policy': 'own_groups'}


<a id="1.5"></a>
### 1.5 🌈 `SpectralSplitter` — Laplacian-eigenmap clustering on an affinity graph

Laplacian-eigenmap spectral clustering on an affinity graph — reliably yields the least train/test overlap among routine structure-based splits.

> 💡 **Advantages:**
> - Minimises inter-cluster similarity by construction, reliably yielding the least train/test overlap among routine structure-based splits.
> - Handles non-convex, elongated regions of chemical space that k-means cuts straight through.
> - The eigenvalue spectrum is a free diagnostic — the spectral gap shows whether the dataset genuinely has that many separable families.

> ⚠️ **Pitfalls:**
> - `O(n²)` affinity construction and a dense-ish eigenproblem cap it near 50,000 molecules.
> - Depends on three coupled choices — graph construction, Laplacian normalisation, and `n_clusters` — none with a chemically principled default.
> - A disconnected affinity graph silently turns spectral clustering into "one cluster per component" — hence the hard error.

| Parameter | Meaning |
|---|---|
| `n_clusters` | number of spectral clusters to form |
| `graph` | affinity graph construction: `knn` or `full` |
| `knn_k` | neighbours per node when `graph="knn"` |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance feeding the affinity graph, e.g. `tanimoto` (default), `dice` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [8]:
splitter = SpectralSplitter(n_clusters=2, graph="knn", knn_k=5, train_size=0.5, test_size=0.5, random_state=0)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "SpectralSplitter")

[SpectralSplitter] train=15 valid=0 test=5 discard=0
  metadata: {'realised_sizes': {'train': 15, 'valid': 0, 'test': 5}, 'n_clusters': 2, 'graph': 'knn', 'nondeterministic_method': True}


<a id="1.6"></a>
### 1.6 📐 `MaxMinSplitter` — greedy maximally-diverse selection

Greedy maximally-diverse selection (MaxMin / Kennard-Stone). The picked set may go to **train** (maximise coverage) or **test** (probe breadth) — opposite experiments sharing one algorithm.

> 💡 **Advantages:**
> - With `picked_goes_to="train"`, builds the most informative training set for a fixed budget.
> - `coverage_radius` is a directly interpretable guarantee — no record sits further than that from a training example.
> - Deterministic apart from a single initial pick, and fully deterministic with `init="kennard_stone"`.

> ⚠️ **Pitfalls:**
> - **The two directions are different experiments and are routinely confused** — never compare numbers across `picked_goes_to` values.
> - Greedy MaxMin chases outliers — the first picks are typically the weirdest molecules in the set, including parse artefacts and fragments.
> - Optimises coverage, not group separation — not a leakage-control split, and shouldn't be described as one.

| Parameter | Meaning |
|---|---|
| `picked_goes_to` | which partition receives the greedily-diverse picks: `train` or `test` |
| `n_picks` | number of records to pick greedily before filling the rest |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance, e.g. `tanimoto` (default), `dice` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [9]:
splitter = MaxMinSplitter(picked_goes_to="train", n_picks=6, train_size=0.5, test_size=0.5, random_state=0)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "MaxMinSplitter")

[MaxMinSplitter] train=6 valid=0 test=10 discard=4
  metadata: {'picked': [5, 10, 17, 19, 16, 13], 'picked_goes_to': 'train', 'init': 'random', 'min_pairwise_distance_in_picked': 0.6521739363670349, 'coverage_radius': 0.625, 'realised_sizes': {'train': 6, 'valid': 0, 'test': 10}}


<a id="1.7"></a>
### 1.7 ↔️ `MaxDissimilaritySplitter` — push train and test to opposite ends

Pushes train and test to opposite regions of chemical space — a clean, reproducible large extrapolation test.

> 💡 **Advantages:**
> - Produces a clean, reproducible large extrapolation — exactly the right test for "can this model reach a region it's never seen?"
> - Only two records are chosen by any rule; everything else follows deterministically, keeping the split easy to describe and audit.
> - `metadata["min_cross_distance"]` quantifies how far apart the two sets actually ended up.

> ⚠️ **Pitfalls:**
> - Deliberately worst-case — it estimates performance on **one specific** extrapolation, not average prospective performance.
> - The whole split hinges on two seed molecules, usually outliers — one badly standardised salt can define the entire experiment.
> - Test and train are contiguous regions, so the test set is chemically homogeneous with strongly correlated errors.

| Parameter | Meaning |
|---|---|
| `seed_pair` | rule for choosing the two seed molecules, e.g. `max_distance` |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance, e.g. `tanimoto` (default), `euclidean` (unbounded metrics are fine here — no cutoff/threshold to interpret as a similarity) |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [10]:
splitter = MaxDissimilaritySplitter(seed_pair="max_distance", train_size=0.5, test_size=0.5, random_state=0)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "MaxDissimilaritySplitter")

[MaxDissimilaritySplitter] train=10 valid=0 test=10 discard=0
  metadata: {'seed_train': 0, 'seed_test': 10, 'seed_distance': 1.0, 'n_tied_seed_pairs': 70, 'min_cross_distance': 0.875, 'realised_sizes': {'train': 10, 'valid': 0, 'test': 10}}


<a id="1.8"></a>
### 1.8 🛰️ `PerimeterSplitter` — hold out the outskirts, train on the dense core

Holds out the outskirts of the distribution; trains on the dense core — directly tests the applicability-domain edge.

> 💡 **Advantages:**
> - Directly tests the applicability-domain edge — the held-out molecules are exactly where a deployed model would be least confident.
> - Completely deterministic, with no seed and no free parameter beyond the metric.
> - The training set stays dense and representative, so training stays stable even though evaluation is hard.

> ⚠️ **Pitfalls:**
> - The test set is enriched in oddities — fragments, salts, dyes, standardisation failures; a poor score may reflect data quality, not model quality.
> - Error bars run large, since the test set is heterogeneous and small in effective size.
> - Not a chemical-novelty guarantee — an outlier can still sit near a training molecule if that's its only near neighbour.

| Parameter | Meaning |
|---|---|
| `pair_rule` | how peripherality is scored, e.g. `outlier_score` |
| `featurizer` | feature representation, e.g. `ecfp4` (default), `maccs`, `physchem` — see [§0.1](#0.1) |
| `metric` | pairwise distance, e.g. `tanimoto` (default), `euclidean` |
| `train_size` / `test_size` | target split fractions |
| `random_state` | seed for reproducibility |

In [11]:
splitter = PerimeterSplitter(pair_rule="outlier_score", train_size=0.6, test_size=0.4, random_state=0)
result = splitter.split_result(SMILES_20)[0]
summarize(result, "PerimeterSplitter")

[PerimeterSplitter] train=12 valid=0 test=8 discard=0
  metadata: {'pair_rule': 'outlier_score', 'n_pairs_used': 0, 'odd_pair_trim': False, 'fallback_filled': 0, 'test_mean_outlier_score': 0.8338360786437988, 'train_mean_outlier_score': 0.6283562779426575, 'realised_sizes': {'train': 12, 'valid': 0, 'test': 8}}


<a id="1.9"></a>
### 1.9 🔁 `LeaveOneClusterOutSplitter` — every cluster gets a turn as the test fold

Each cluster (from a caller-supplied `clusterer`) takes a turn as the test fold — yields a per-cluster error distribution instead of one number.

> 💡 **Advantages:**
> - Yields a **per-cluster error distribution** instead of one number — you learn which regions of chemical space the model fails in.
> - Every record is tested exactly once (when `max_folds=None`), so the aggregate is an honest whole-dataset estimate.
> - Composes with every scaffold/similarity/embedding grouping, so the same protocol answers many "generalise across X?" questions.

> ⚠️ **Pitfalls:**
> - Cluster sizes are uneven, so per-fold scores come from wildly different sample sizes — report both macro- and micro-averages.
> - Many small clusters make the fold count explode and the run expensive; `max_folds` caps it but biases the aggregate.
> - A single-cluster test fold with 3 records can't support ROC-AUC or a meaningful R².

| Parameter | Meaning |
|---|---|
| `clusterer` | splitter/clusterer instance supplying the group labels (here, a `ButinaSplitter`) |
| `max_folds` | cap on the number of folds actually run |

In [12]:
splitter = LeaveOneClusterOutSplitter(
    clusterer=ButinaSplitter(cutoff=0.15, random_state=0), max_folds=None,
)
results = splitter.split_result(SMILES_20)
print(f"{len(results)} folds")
for i, r in enumerate(results):
    summarize(r, f"  fold {i}")

12 folds
[  fold 0] train=15 valid=0 test=5 discard=0
  metadata: {'fold_index': 0, 'cluster_id': 0, 'cluster_size': 5, 'n_clusters': 12, 'clusters_never_tested': []}
[  fold 1] train=15 valid=0 test=5 discard=0
  metadata: {'fold_index': 1, 'cluster_id': 1, 'cluster_size': 5, 'n_clusters': 12, 'clusters_never_tested': []}
[  fold 2] train=19 valid=0 test=1 discard=0
  metadata: {'fold_index': 2, 'cluster_id': 2, 'cluster_size': 1, 'n_clusters': 12, 'clusters_never_tested': []}
[  fold 3] train=19 valid=0 test=1 discard=0
  metadata: {'fold_index': 3, 'cluster_id': 3, 'cluster_size': 1, 'n_clusters': 12, 'clusters_never_tested': []}
[  fold 4] train=19 valid=0 test=1 discard=0
  metadata: {'fold_index': 4, 'cluster_id': 4, 'cluster_size': 1, 'n_clusters': 12, 'clusters_never_tested': []}
[  fold 5] train=19 valid=0 test=1 discard=0
  metadata: {'fold_index': 5, 'cluster_id': 5, 'cluster_size': 1, 'n_clusters': 12, 'clusters_never_tested': []}
[  fold 6] train=19 valid=0 test=1 discard=

<a id="1.10"></a>
### 1.10 ⚖️ `BalancedMultiTaskSplitter` — balance every task's label coverage, not just record counts

Balanced multi-task cluster assignment: assigns whole clusters to folds so every task gets an acceptable train/test ratio and label balance — the practical answer to sparse multi-task matrices.

> 💡 **Advantages:**
> - The practical answer to sparse multi-task matrices, where naive cluster splitting can leave some targets with zero test actives.
> - Balance is a *constraint*, not a hope — if the requested balance is impossible, the splitter says so and names the binding task.
> - `per_task_fold_counts` gives a complete audit of what every task got.

> ⚠️ **Pitfalls:**
> - Infeasibility is common on real sparse matrices; `on_infeasible="relax"` is the pragmatic escape, but read `metadata["tolerance_used"]`.
> - Solve time grows quickly with cluster count; the `dust` merge that keeps it tractable changes the grouping invisibly.
> - Balancing on label statistics chooses the split partly using the labels — a mild, usually-acceptable form of design leakage.

| Parameter | Meaning |
|---|---|
| `train_size` / `test_size` | target split fractions |
| `tolerance` | allowed deviation from the per-task target balance |
| `random_state` | seed for reproducibility |

In [13]:
rng = np.random.default_rng(0)
n = 60
smiles = [_FAMILY_A[i % 10] if i % 2 == 0 else _FAMILY_B[i % 10] for i in range(n)]
y = rng.random((n, 3))
mask = rng.random((n, 3)) < 0.7
y[~mask] = np.nan

splitter = BalancedMultiTaskSplitter(train_size=0.7, test_size=0.3, tolerance=0.3, random_state=0)
result = splitter.split_result(smiles, y=y)[0]
summarize(result, "BalancedMultiTaskSplitter")

[BalancedMultiTaskSplitter] train=42 valid=0 test=18 discard=0
  metadata: {'n_clusters': 7, 'n_tasks': 3, 'solver': 'bnb', 'solver_status': 'optimal', 'objective': 0.16742344176994567, 'tolerance_used': 0.3, 'per_task_fold_counts': [[25.0, 12.0], [30.0, 13.0], [26.0, 11.0]], 'realised_sizes': {'train': 42, 'valid': 0, 'test': 18}}


`y` is a 60x3 sparse multi-task matrix (70% coverage per task) — the splitter keeps every task's train/test balance within `tolerance` of the target.

### 🏁 Wrap-up

All 10 similarity splitters share the same call shape (`splitter.split_result(X, y=...)`), so swapping one for another in an evaluation pipeline is a one-line change — the difference is entirely in *how hard* and *how they fail*. Start from `butina` as a sane default, escalate to `similarity_threshold` when you need an explicit, checkable novelty guarantee, and reach for `leave_one_cluster_out` or `balanced_multi_task` when one train/test split isn't enough evidence.

Whichever splitter you pick, don't forget [§0.1](#0.1): `featurizer` and `metric` are not tuning knobs to leave at their defaults — they define what "similar" means for your data, and belong in any report of results right alongside the splitter's name.